# Huấn luyện Mô hình Nhánh 1 Tay (One-Hand Branch Classifier)
Notebook này thực hiện huấn luyện mô hình cho cử chỉ 1 tay (58 đặc trưng x 30 khung hình), đánh giá F1-Score, vẽ Confusion Matrix và xuất tệp `one_hand_classifier.tflite`.


In [1]:
import re
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

from models.sequence_classifier import convert_to_tflite

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.13.0


In [2]:
# 1. Nạp dữ liệu từ data/one_hand/
data_dir = "data/one_hand"
X, y = [], []
label_names = []

label_csv = os.path.join(data_dir, "keypoint_classifier_label.csv")
if os.path.exists(label_csv):
    with open(label_csv, "r", encoding="utf-8") as f:
        label_names = [line.strip() for line in f if line.strip()]
else:
    label_names = ["Like", "OK", "Peace", "Point", "Stop"]

print(f"Danh sách nhãn Nhánh 1 Tay ({len(label_names)} lớp): {label_names}")

# Nạp tệp .npy nếu có (hỗ trợ cả tên nhãn gốc và tên an toàn trên Windows)
for idx, label in enumerate(label_names):
    safe_label = re.sub(r'[\\/*?:"<>|]', '-', label).strip('. ')
    folder = os.path.join(data_dir, safe_label)
    if not os.path.exists(folder):
        folder = os.path.join(data_dir, label)
    files = glob.glob(os.path.join(folder, "*.npy"))
    for fpath in files:
        arr = np.load(fpath)
        if arr.shape == (30, 58):
            X.append(arr)
            y.append(idx)

# Tạo dữ liệu giả lập nếu chưa có mẫu thực tế
if len(X) == 0:
    print("[DEMO] Chưa phát hiện dữ liệu thực tế. Đang tạo 150 mẫu giả lập để chạy thử...")
    for c in range(len(label_names)):
        for _ in range(30):
            sample = np.random.normal(c * 0.2, 0.1, (30, 58)).astype(np.float32)
            X.append(sample)
            y.append(c)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
print(f"X shape: {X.shape}, y shape: {y.shape}")


Danh sách nhãn Nhánh 1 Tay (10 lớp): ['xin chào', 'tốt', 'tôi', 'bạn', 'chúng tôi', 'bọn họ', 'có / đúng', 'không', 'yêu', 'dừng lại']
X shape: (590, 30, 58), y shape: (590,)


In [3]:
# 2. Phân chia Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Dựng mô hình Bi-LSTM cho Nhánh 1 Tay
input_dim = X_train.shape[2]
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, input_dim)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True, unroll=True)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, unroll=True)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(label_names), activation="softmax")
], name="OneHandClassifierModel")

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=16)


Epoch 1/25

30/30 [==============================] - 13s 61ms/step - loss: 1.4600 - accuracy: 0.5699 - val_loss: 0.7320 - val_accuracy: 0.6949
Epoch 2/25

30/30 [==============================] - 0s 11ms/step - loss: 0.3523 - accuracy: 0.9131 - val_loss: 0.1905 - val_accuracy: 0.9661
Epoch 3/25

30/30 [==============================] - 0s 11ms/step - loss: 0.0958 - accuracy: 0.9725 - val_loss: 0.0842 - val_accuracy: 0.9746
Epoch 4/25

30/30 [==============================] - 0s 11ms/step - loss: 0.0606 - accuracy: 0.9831 - val_loss: 0.1298 - val_accuracy: 0.9746
Epoch 5/25

30/30 [==============================] - 0s 10ms/step - loss: 0.0570 - accuracy: 0.9788 - val_loss: 0.0613 - val_accuracy: 0.9831
Epoch 6/25

30/30 [==============================] - 0s 10ms/step - loss: 0.0337 - accuracy: 0.9873 - val_loss: 0.0256 - val_accuracy: 0.9915
Epoch 7/25

30/30 [==============================] - 0s 10ms/step - loss: 0.0087 - accuracy: 1.0000 - val_loss: 0.0317 - val_accuracy: 0.9915
Epoch

In [4]:
# 4. Đánh giá & Confusion Matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=label_names))

# 5. Xuất TFLite Float16 cho Nhánh 1 Tay
out_tflite = "model/one_hand_classifier/one_hand_classifier.tflite"
convert_to_tflite(model, output_path=out_tflite, float16_quantization=True)


4/4 [==============================] - 2s 4ms/step


ValueError: Number of classes, 9, does not match size of target_names, 10. Try specifying the labels parameter